# Tool 8bis: Automatic epoch rejection, Curry twin (channel-first, then epoch)

*Curry twin: functionally identical to the EDF tool 8bis. 8bis reads only tool-7 `*_all-epo.fif` + `*_epoch_channel_rejection.tsv`, which the Curry tool 7 writes in the same MNE/TSV format, so no code differs. Generated by `tools_curry/_make_tool8bis_curry.py`. Do not hand-edit: edit the EDF notebook and re-run the generator.*

Automatically cleans the epochs that **7_preprocessing** flagged, using a **channel-first then epoch**
rule (à la PREP / FASTER): a globally-bad channel is dropped *before* the epoch vote, so it no longer
condemns every epoch it appears in.

**Per participant:** ① a **channel is dropped** when flagged in more than *Channel reject %* of the
epochs of the **reference stages** → ② **every epoch** (whatever its stage) is rejected when flagged in
more than *Epoch reject %* of the remaining **good** channels → ③ the decision is written as two tables,
`*_epoch_decision.tsv` (every epoch: in scope = always True, rejected, why) and `*_channel_decision.tsv` (every channel: dropped, why), to `rejection_auto/`
**beside the preprocessed-epochs folder**, and the reports (decision record, HTML, global summary) to
`reports_rejection_auto/` **beside the reports folder**: following the toolkit's `derivatives/` (data) vs
`reports_*` (reports) split. No epoch is copied: tool 9 reads tool 7's `*_all-epo.fif` and these tables,
so a decision can be changed (re-run with other thresholds) without rewriting any signal.

**Inputs (read-only):** two folders are selected **explicitly**, the **preprocessed-epochs folder** (holding the
tool-7 `{file_id}_all-epo.fif`, e.g. `derivatives/preproc_epo`) and the **reports folder** (holding
`{file_id}_epoch_channel_rejection.tsv`, e.g. `reports_preproc`). Picking them explicitly (rather than
scanning one root recursively) keeps several tool-7 runs apart when their outputs have been renamed/versioned
(`preproc_epo_v1`, `reports_preproc_v2`, …). Tool-7 outputs are never modified. No channel interpolation:
bad channels are dropped (interpolation is a later step).

**Workflow:** ① Select the preprocessed-epochs + reports folders and **Scan** → ② set thresholds + reference
stages → ③ Run.

**The decision covers every epoch.** Which stages to analyse is chosen later (tool 9, statistics), so one
decision serves every analysis. The reference stages only decide which epochs *judge* the channels (and the
participant exclusion), so that wake movement / EMG artefacts do not drop a channel that is clean in sleep.

In [ ]:
import os, sys, io, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ipyfilechooser import FileChooser
import mne
mne.set_log_level('ERROR')

# Reuse the shared tool-8 library (find_participants, save_report_html, custom-stage styling). Search
# several locations so it is found whether Voila is launched from the repo root (lib in tools/) or from
# tools_curry/ (the Curry twin lives there; the shared lib stays in ../tools).
_here = os.getcwd()
_lib_candidates = [_here, os.path.join(_here, 'tools'),
                   os.path.join(os.path.dirname(_here), 'tools'), os.path.join(_here, '..', 'tools')]
for _cand in _lib_candidates:
    if os.path.isfile(os.path.join(_cand, 'qc_rejected_epochs_lib.py')):
        _cand = os.path.abspath(_cand)
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break
import qc_rejected_epochs_lib as L
import participant_selection_lib as PSL     # exclusion registry + participant selector (same folder)
# The decision rule, its tables and figures, shared with tool 0: tools/autoreject_lib.py (same folder as
# qc_rejected_epochs_lib). Edit the library, never a copy.
from autoreject_lib import (
    read_event_counts, build_pair_matrix, auto_reject_decision, decision_summary_row, read_tool7_bad_channels,
    build_epoch_decision, build_channel_decision, wrap_scroll, decision_overview_html, plot_decision_heatmap,
    ordered_stages, default_reference_stages, event_exclusion_reason, registry_reason, AASM_STAGES,
    DEFAULT_CHANNEL_REJECT_PCT, DEFAULT_EPOCH_REJECT_PCT,
)

# The decision tables live beside the data (derivatives/rejection_auto/<subtree>/), the reports in
# reports_rejection_auto/: tool 9 reads tool 7's *_all-epo.fif + these tables, so a decision can be changed
# without rewriting any epochs.
DECISION_DIRNAME = 'rejection_auto'

S = {}   # shared state across sections


def registry_folder():
    """Where the exclusion registry lives: the study folder when picked, else the folder holding the
    derivatives/ the preprocessed-epochs folder sits in."""
    if fc_data.selected_path:
        return Path(fc_data.selected_path)
    return PSL.find_data_folder(S.get('raw_root'))


## Section 1 - Select the folders

Optionally set the **study folder** first (holds `derivatives/` + `reports_preproc/`, as in tool 7):
it just pre-points the next two choosers. Then pick the **preprocessed-epochs folder** (holds the tool-7
`{file_id}_all-epo.fif`, e.g. `derivatives/preproc_epo`) and the **reports folder** (holds
`{file_id}_epoch_channel_rejection.tsv`, e.g. `reports_preproc`), and click **Scan**. Selecting the
two folders explicitly keeps renamed/versioned tool-7 runs apart. The decision tables are written to
`rejection_auto/` (beside the raw folder). The reports go to `reports_rejection_auto/` (beside the reports
folder).

In [ ]:
fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = ('<b>Study folder</b> (holds <code>derivatives/</code> + '
                 '<code>reports_preproc/</code>), optional, pre-points the two folders below:')

fc_raw = FileChooser(os.getcwd())
fc_raw.show_only_dirs = True
fc_raw.title = '<b>Preprocessed-epochs folder</b> (holds the tool-7 <code>*_all-epo.fif</code>):'

fc_reports = FileChooser(os.getcwd())
fc_reports.show_only_dirs = True
fc_reports.title = '<b>Reports folder</b> (holds <code>*_epoch_channel_rejection.tsv</code>):'

btn_scan = widgets.Button(description='Scan', button_style='primary', icon='search')
lbl_scan = widgets.HTML('<i>Optionally set the study folder, pick the preprocessed-epochs and reports folders, then Scan.</i>')


def _on_data(chooser):
    # Convenience: point fc_raw into <data>/derivatives (where preproc_epo* live) and fc_reports at <data>
    # (the parent of reports_preproc*). The user still picks the specific (possibly versioned) folder.
    try:
        data = fc_data.selected_path
        if not data:
            return
        data = Path(data)
        raw_start = data / 'derivatives' if (data / 'derivatives').is_dir() else data
        fc_raw.reset(path=str(raw_start))
        fc_raw.title = '<b>Preprocessed-epochs folder</b> (holds the tool-7 <code>*_all-epo.fif</code>):'
        fc_reports.reset(path=str(data))
        fc_reports.title = '<b>Reports folder</b> (holds <code>*_epoch_channel_rejection.tsv</code>):'
        lbl_scan.value = ('<i>Study folder set: pick the preprocessed-epochs folder (inside '
                          '<code>derivatives/</code>) and the reports folder, then Scan.</i>')
    except Exception as e:
        lbl_scan.value = f'<span style="color:#c62828">Study folder error: {e}</span>'


def _scan(b):
    try:
        raw, reports = fc_raw.selected_path, fc_reports.selected_path
        if not raw or not reports:
            lbl_scan.value = '<span style="color:#c62828">Select both folders first.</span>'; return
        raw_root, reports_root = Path(raw), Path(reports)
        parts = L.find_participants(raw_root)
        # Locate each participant's flags TSV + the optional event sidecars in the chosen reports
        # folder (once; reused by the run). Also collect the methods present and the event types.
        tsv_by_fid = {}
        flags_by_fid = {}      # {fid: _event_epoch_flags.tsv path or None}
        counts_by_fid = {}     # {fid: _event_counts.tsv path or None}
        stages = set(); methods_present = set(); event_types = set()
        type_counts = {}       # {event_type: [n_events per file that has it]} -> mean/median hints
        for p in parts:
            fid = p['file_id']
            tsv = next(reports_root.rglob(f'{fid}_epoch_channel_rejection.tsv'), None)
            tsv_by_fid[fid] = tsv
            if tsv is not None:
                try:
                    cols = pd.read_csv(tsv, sep='\t', nrows=0).columns
                    methods_present.update(m for m in L.METHOD_ORDER if ('flag_' + m) in cols)
                    stages.update(pd.read_csv(tsv, sep='\t', usecols=['stage'])['stage'].astype(str).unique())
                except Exception:
                    pass
            ef = next(reports_root.rglob(f'{fid}_event_epoch_flags.tsv'), None)
            flags_by_fid[fid] = ef
            if ef is not None:
                try:
                    event_types.update(c[len('evt_'):] for c in pd.read_csv(ef, sep='\t', nrows=0).columns
                                       if c.startswith('evt_'))
                except Exception:
                    pass
            ec = next(reports_root.rglob(f'{fid}_event_counts.tsv'), None)
            counts_by_fid[fid] = ec
            if ec is not None:
                try:
                    ecdf = pd.read_csv(ec, sep='\t')
                    event_types.update(ecdf['event_type'].astype(str).unique())
                    for _, r in ecdf.iterrows():
                        type_counts.setdefault(str(r['event_type']), []).append(int(r['n_events']))
                except Exception:
                    pass
        ordered = ordered_stages(stages)
        S['raw_root'] = raw_root; S['reports_root'] = reports_root
        S['parts'] = parts; S['tsv_by_fid'] = tsv_by_fid
        S['flags_by_fid'] = flags_by_fid; S['counts_by_fid'] = counts_by_fid
        # ---- Stage checkboxes (default ticked: N2/N3/R, else all) ----
        prefer = default_reference_stages(ordered)
        S['stage_checks'] = {}
        boxes = []
        for st in ordered:
            cb = widgets.Checkbox(value=(st in prefer), description=st,
                                  indent=False, layout=widgets.Layout(width='75px'))
            S['stage_checks'][st] = cb
            boxes.append(cb)
        stage_box.children = boxes
        # ---- Flagging-method checkboxes (all ON by default = today's reject_any) ----
        S['method_checks'] = {}
        mboxes = []
        for m in [m for m in L.METHOD_ORDER if m in methods_present]:
            cb = widgets.Checkbox(value=True, description=L.METHOD_LABEL.get(m, m),
                                  indent=False, layout=widgets.Layout(width='120px'))
            S['method_checks'][m] = cb
            mboxes.append(cb)
        method_box.children = mboxes
        # ---- Event-type rows (checkbox + per-type exclusion threshold), nested under the 'event' method ----
        S['event_type_rows'] = {}
        erows = []
        for t in sorted(event_types):
            cbt = widgets.Checkbox(value=True, description=t, indent=False,
                                   layout=widgets.Layout(width='190px'))
            thr = widgets.IntText(value=0, layout=widgets.Layout(width='80px'))
            # Per-file event-count hint across the files that HAVE this type (guides the threshold).
            vals = type_counts.get(t, [])
            hint = (f'&nbsp;<small style="color:#666;">mean {np.mean(vals):.1f} (median '
                    f'{np.median(vals):g}) events/file, n={len(vals)}</small>') if vals else ''
            erows.append(widgets.HBox([cbt, widgets.HTML('exclude participant if &gt;'), thr,
                                       widgets.HTML('events (0 = off)'), widgets.HTML(hint)]))
            S['event_type_rows'][t] = (cbt, thr)
        event_types_box.children = erows
        # Reveal the event-type rows only when the 'event' method is ticked (initial display from its value).
        ev_master = S['method_checks'].get('event')
        if ev_master is not None:
            def _toggle_events(change=None):
                event_section.layout.display = '' if ev_master.value else 'none'
            ev_master.observe(_toggle_events, names='value')
            _toggle_events()
        else:
            event_section.layout.display = 'none'   # no event flagging in any scanned file
        n_tsv = sum(1 for t in tsv_by_fid.values() if t is not None)
        colour = '#2e7d32' if (parts and n_tsv == len(parts)) else ('#c62828' if not parts else '#e69500')
        msg = f'{len(parts)} participant(s) in the raw folder; {n_tsv} with a matching TSV in the reports folder.'
        if parts and n_tsv < len(parts):
            msg += ' &#9888; some participants have no TSV here (version mismatch?).'
        # Participants for the selector: done = decision record (reports) + epoch decision (data) on disk
        reports_out = reports_root.parent / 'reports_rejection_auto'
        done = []
        for p in parts:
            try:
                subtree = Path(p['folder']).relative_to(raw_root)
            except Exception:
                subtree = Path('.')
            if ((reports_out / subtree / f"{p['file_id']}_autoreject_decision.tsv").exists()
                    and (raw_root.parent / DECISION_DIRNAME / subtree / f"{p['file_id']}_epoch_decision.tsv").exists()):
                done.append(p['file_id'])
        reg, reg_warn = PSL.load_registry(registry_folder())
        participant_selector.set_participants(
            [p['file_id'] for p in parts], done=done,
            excluded={s['file_id']: s['reason'] for s in PSL.effective_status(reg).values() if s['excluded']})
        if reg_warn:
            msg += f' &#9888; {reg_warn}'
        stale = (PSL.stale_tables_html(PSL.STALE_RERUN)
                 if PSL.registry_newer_than(registry_folder(), reports_out / 'global_autoreject_summary.tsv') else '')
        lbl_scan.value = (f'<span style="color:{colour}">{msg}</span>'
                          f'<br><small>decisions &rarr; {raw_root.parent / DECISION_DIRNAME}'
                          f'<br>reports &rarr; {reports_out}</small>' + stale)
    except Exception as e:
        lbl_scan.value = f'<span style="color:#c62828">Scan error: {e}</span>'


fc_data.register_callback(_on_data)
btn_scan.on_click(_scan)
display(widgets.VBox([fc_data, fc_raw, fc_reports, btn_scan, lbl_scan]))


## Section 2 - Methods, thresholds & reference stages

A **channel** is dropped when flagged in more than *Channel reject %* of the epochs of the **reference
stages**. Then **every epoch**, whatever its stage, is rejected when flagged in more than *Epoch reject %*
of the remaining **good** channels. Both default to 20% and are editable. The reference stages (default
N2/N3/R) judge the channels and the participant exclusion; the decision itself covers every epoch
(`in_scope = True` everywhere), the stages to analyse being chosen later (tool 9, statistics).

In [ ]:
ft_ch_pct = widgets.FloatText(value=DEFAULT_CHANNEL_REJECT_PCT, description='Channel reject > (%)',
                              style={'description_width': '150px'}, layout=widgets.Layout(width='250px'))
ft_ep_pct = widgets.FloatText(value=DEFAULT_EPOCH_REJECT_PCT, description='Epoch reject > (%)',
                              style={'description_width': '150px'}, layout=widgets.Layout(width='250px'))
# All three boxes below are populated when you Scan (Section 1): one checkbox per detected stage,
# one per flagging method present, and one row (checkbox + exclusion threshold) per event type.
stage_box = widgets.Box([], layout=widgets.Layout(display='flex', flex_flow='row wrap',
                                                  width='100%'))
method_box = widgets.Box([], layout=widgets.Layout(display='flex', flex_flow='row wrap', width='100%'))
# Nested under the 'event' method checkbox: revealed only when 'event' is ticked (wired at Scan time).
event_types_box = widgets.VBox([], layout=widgets.Layout(margin='2px 0 0 25px'))
event_section = widgets.VBox([
    widgets.HTML('<b>Event types</b>: tick the types allowed to reject an epoch; set a '
                 'threshold to <b>exclude the whole participant</b> when the file has more events of '
                 'that type than the threshold (0 = no exclusion). Needs the tool-7 '
                 '<code>*_event_epoch_flags.tsv</code> / <code>*_event_counts.tsv</code> sidecars; '
                 'files preprocessed before this feature fall back to the whole <code>flag_event</code> '
                 '(all event types, no sub-selection).'),
    event_types_box,
])

display(widgets.VBox([
    ft_ch_pct, ft_ep_pct,
    widgets.HTML('<b>Flagging methods</b> (populated at Scan): tick the tool-7 methods that '
                 'drive the rejection: all on reproduces the previous behaviour, but after inspecting '
                 'the tool-7 flagging reports you can drop one that looks unhelpful. Ticking '
                 '<b>event</b> reveals the event types below.'),
    method_box, event_section,
    widgets.HTML('<b>Reference stages</b>: the stages that judge the channels (channel badness) and '
                 'the participant exclusion. The epoch rule is then applied to <b>every</b> epoch, '
                 'whatever its stage (the stages to analyse are chosen in tool 9):'),
    stage_box,
]))


## Section 3 - Run

Loops over the participants chosen below (all, or a subset). `{file_id}_epoch_decision.tsv` +
`{file_id}_channel_decision.tsv` go to `rejection_auto/<subtree>/` (beside the raw folder). `{file_id}_autoreject_decision.tsv` +
`{file_id}_autoreject_report.html` + the rebuilt `global_autoreject_summary.tsv` go to
`reports_rejection_auto/` (beside the reports folder). A **summary table** is shown at the end. A
participant is
**skipped** if it already has both its epoch decision and its decision record (uncheck *Skip* to reprocess).
Participants where **every channel** (or every reference-stage epoch) would be rejected still get their tables and
a report (shown as **100% rejected** in the summary) and are **excluded automatically** in the exclusion
registry (`8bis_auto`), like a participant over an event-count threshold. A re-run that no longer
excludes them lifts that exclusion.

In [ ]:
participant_selector = PSL.ParticipantSelector()       # all participants / a subset (+ skip)
cb_skip = participant_selector.cb_skip
btn_run = widgets.Button(description='Run automatic rejection', button_style='success', icon='play')
prog = widgets.IntProgress(description='Participants:', min=0, max=1, value=0,
                           layout=widgets.Layout(width='430px'))
lbl_prog = widgets.HTML(layout=widgets.Layout(margin='0 0 0 8px'))   # current participant (i/N)
out_run = widgets.Output()


def _rebuild_global_summary(auto_root):
    files = sorted(Path(auto_root).rglob('*_autoreject_decision.tsv'))
    rows = []
    for f in files:
        try:
            rows.append(pd.read_csv(f, sep='\t', dtype={'file_id': str}))
        except Exception:
            pass
    if not rows:
        return pd.DataFrame()
    summary = pd.concat(rows, ignore_index=True)
    # flag (never drop) the participants excluded in the registry (5bis / tools 7, 8, 8bis)
    reg, _warn = PSL.load_registry(registry_folder())
    return PSL.add_exclusion_columns(summary, PSL.effective_status(reg))


def run_autoreject(b):
    with out_run:
        clear_output(wait=True)
        try:
            if 'raw_root' not in S:
                print('Select the raw + reports folders and click Scan first (Section 1).'); return
            raw_root = Path(S['raw_root']); reports_root = Path(S['reports_root'])
            soi = [st for st, cb in S.get('stage_checks', {}).items() if cb.value]
            if not soi:
                print('Tick at least one stage of interest (Section 2).'); return
            ch_pct = float(ft_ch_pct.value) / 100.0
            ep_pct = float(ft_ep_pct.value) / 100.0
            skip = cb_skip.value
            if participant_selector.warning_html():      # a subset with already-processed participants
                display(HTML(participant_selector.warning_html()))
            chosen = {PSL.id_key(f) for f in participant_selector.chosen_ids()}
            reg_folder = registry_folder()
            # Section-2 flagging-method + event sub-selection.
            methods_sel = [m for m, cb in S.get('method_checks', {}).items() if cb.value]
            event_master_on = ('event' in methods_sel)
            event_types_sel = [t for t, (cb, thr) in S.get('event_type_rows', {}).items() if cb.value]
            # Per-type exclusion thresholds: only ticked types with a positive threshold are evaluated.
            event_thresholds = {t: int(thr.value) for t, (cb, thr) in S.get('event_type_rows', {}).items()
                                if cb.value and int(thr.value) > 0}
            if not methods_sel:
                print('Tick at least one flagging method (Section 2).'); return
            flags_by_fid = S.get('flags_by_fid', {}); counts_by_fid = S.get('counts_by_fid', {})
            parts = [p for p in S.get('parts', []) if PSL.id_key(p['file_id']) in chosen]
            tsv_by_fid = S.get('tsv_by_fid', {})
            # DATA (the epoch / channel decision tables) beside the preprocessed-epochs folder; REPORTS (decision
            # record + html + globals) beside the reports folder: the toolkit's derivatives/ vs reports_* split.
            reports_out = reports_root.parent / 'reports_rejection_auto'
            prog.max = max(1, len(parts)); prog.value = 0
            failed = []; saved = 0; all_rejected = 0; skipped = 0; excluded = 0
            for i, p in enumerate(parts):
                fid = p['file_id']
                lbl_prog.value = f'<b>{fid}</b> ({i + 1}/{len(parts)})'
                try:
                    try:
                        subtree = Path(p['folder']).relative_to(raw_root)   # participant subtree in the raw folder
                    except Exception:
                        subtree = Path('.')
                    reports_dir = reports_out / subtree                     # holds the decision tsv + html
                    decision_dir = raw_root.parent / DECISION_DIRNAME / subtree   # epoch/channel decisions
                    decision_tsv = reports_dir / f'{fid}_autoreject_decision.tsv'
                    epoch_dec_tsv = decision_dir / f'{fid}_epoch_decision.tsv'
                    if skip and epoch_dec_tsv.exists() and decision_tsv.exists():
                        skipped += 1; prog.value += 1; continue
                    tsv = tsv_by_fid.get(fid) or next(reports_root.rglob(f'{fid}_epoch_channel_rejection.tsv'), None)
                    if tsv is None:
                        failed.append((fid, 'no _epoch_channel_rejection.tsv in the reports folder'))
                        prog.value += 1; continue
                    ef_path = flags_by_fid.get(fid) or next(reports_root.rglob(f'{fid}_event_epoch_flags.tsv'), None)
                    ec_path = counts_by_fid.get(fid) or next(reports_root.rglob(f'{fid}_event_counts.tsv'), None)
                    piv, stage_by_epoch, pm_meta = build_pair_matrix(
                        tsv, methods=methods_sel, event_master=event_master_on,
                        event_types=event_types_sel, event_flags_path=ef_path)
                    if pm_meta.get('warn'):
                        print(f'  ⚠ {fid}: {pm_meta["warn"]}')
                    methods_used_str = '+'.join(pm_meta['used_methods']) or '(none)'
                    event_types_used_str = '+'.join(pm_meta['used_event_types'])
                    stages_present = set(stage_by_epoch.dropna().astype(str).unique())
                    soi_p = [s for s in soi if s in stages_present]
                    if not soi_p:
                        failed.append((fid, 'none of the selected stages present')); prog.value += 1; continue
                    dec = auto_reject_decision(piv, stage_by_epoch, soi_p, ch_pct, ep_pct)
                    tool7_bads = read_tool7_bad_channels(p['folder'], fid)
                    # Per-event participant exclusion: drop the WHOLE participant when the file has more
                    # events of a ticked type than its threshold (raw event count from _event_counts.tsv).
                    excl_reason = ''
                    if event_master_on and event_thresholds:
                        counts = read_event_counts(ec_path) if ec_path is not None else {}
                        excl_reason = event_exclusion_reason(counts, event_thresholds)
                    # An all-rejected participant (every channel dropped, or every reference-stage epoch rejected)
                    # is a valid extreme outcome: its tables + report are written like any other (100 %
                    # rejected in the summary) and it is excluded in the registry below.
                    has_output = ((not excl_reason) and bool(dec['good_channels'])
                                  and dec['n_rejected_ref'] < dec['n_ref'])
                    reports_dir.mkdir(parents=True, exist_ok=True)
                    # --- DATA first: the epoch + channel decision tables (what tool 9 joins to the epochs) ---
                    decision_dir.mkdir(parents=True, exist_ok=True)
                    build_channel_decision(fid, dec, ch_pct, tool7_bads).to_csv(
                        decision_dir / f'{fid}_channel_decision.tsv', sep='\t', index=False)
                    build_epoch_decision(fid, dec, piv).to_csv(epoch_dec_tsv, sep='\t', index=False)
                    # --- decision TSV (always: feeds the summary, incl. all-rejected / excluded participants) ---
                    decision_summary_row(fid, dec, soi_p, ch_pct, ep_pct,
                                         methods_used=methods_used_str, event_types_used=event_types_used_str,
                                         excluded=bool(excl_reason), exclude_reason=excl_reason).to_csv(
                        decision_tsv, sep='\t', index=False)
                    # --- the exclusion registry: this tool's automatic exclusions for this participant
                    # (replacing its previous ones, so a re-run with other thresholds can lift them) ---
                    auto_reason = registry_reason(excl_reason, dec)
                    try:
                        PSL.write_registry_rows(reg_folder, '8bis_auto',
                                                [{'file_id': fid, 'reason': auto_reason}] if auto_reason else [],
                                                replace_ids=[fid])
                    except Exception as reg_e:
                        print(f'  ⚠ {fid}: exclusion registry not updated ({reg_e}).')
                    # --- per-participant report (always, non-fatal) ---
                    try:
                        custom = [s for s in stages_present if s not in AASM_STAGES]
                        fig = plot_decision_heatmap(piv, dec, fid, custom)
                        html = decision_overview_html(fid, dec, soi_p, ch_pct, ep_pct,
                                                      methods_used=methods_used_str,
                                                      event_types_used=event_types_used_str,
                                                      exclude_reason=excl_reason)
                        L.save_report_html(reports_dir / f'{fid}_autoreject_report.html',
                                           f'{fid} - automatic rejection', [('Decision', fig)], html)
                    except Exception as rep_e:
                        print(f'  ⚠ {fid}: report failed ({rep_e}).')
                    if excl_reason:
                        excluded += 1
                    elif has_output:
                        saved += 1
                    else:
                        all_rejected += 1
                except Exception as e:
                    failed.append((fid, str(e)))
                prog.value += 1
            lbl_prog.value = f'done ({len(parts)}/{len(parts)})'
            # --- global summary (rebuilt from disk) + failures --- (all in the reports folder)
            summary = _rebuild_global_summary(reports_out)
            if not summary.empty:
                summary = summary.sort_values('file_id')
                reports_out.mkdir(parents=True, exist_ok=True)
                summary.to_csv(reports_out / 'global_autoreject_summary.tsv', sep='\t', index=False)
            if failed:
                reports_out.mkdir(parents=True, exist_ok=True)
                pd.DataFrame(failed, columns=['file_id', 'reason']).to_csv(
                    reports_out / 'autoreject_failed.tsv', sep='\t', index=False)
            print(f'Done. decided={saved}  all-rejected={all_rejected}  '
                  f'excluded (event threshold)={excluded}  skipped={skipped}  failed={len(failed)}')
            print(f'  decisions  -> {raw_root.parent / DECISION_DIRNAME}')
            print(f'  reports    -> {reports_out}')
            if failed:
                print('Failed (could not process):')
                for fid, why in failed:
                    print(f'  ⚠ {fid}: {why}')
            # --- end-of-run summary table + global report ---
            if not summary.empty:
                # Compact inline table (the full table stays in the global + per-participant reports).
                disp = summary[['file_id', 'n_channels', 'n_channels_rejected',
                                'n_epochs', 'n_epochs_rejected']].copy()
                disp['n_channels_rejected'] = summary.apply(
                    lambda r: (f"{int(r['n_channels_rejected'])} "
                               f"({100 * r['n_channels_rejected'] / r['n_channels']:.1f}%)")
                    if r['n_channels'] else str(int(r['n_channels_rejected'])), axis=1)
                disp['n_epochs_rejected'] = summary.apply(
                    lambda r: f"{int(r['n_epochs_rejected'])} ({r['pct_epochs_rejected']:.1f}%)", axis=1)
                # the share over the reference stages (absent from records written before this column)
                if 'pct_epochs_rejected_ref' in summary.columns:
                    disp['% rejected (reference stages)'] = summary['pct_epochs_rejected_ref'].map(
                        lambda v: '' if pd.isna(v) else f'{v:.1f}%')
                # Show the exclusions of the registry (5bis criteria, manual, tool 7/8, this tool), if any.
                if summary[PSL.COL_EXCLUDED].any():
                    disp['excluded'] = summary[PSL.COL_REASON].fillna('').values
                display(HTML('<h3>Summary - automatic rejection</h3>' + disp.to_html(index=False)))
                try:
                    # a group figure: the excluded participants are left out (and named in the title)
                    kept = summary[~summary[PSL.COL_EXCLUDED]]
                    n_out = len(summary) - len(kept)
                    fig, ax = plt.subplots(figsize=(max(6, min(14, 2 + 0.5 * len(kept))), 3.6),
                                           layout='constrained')
                    # over the reference stages: comparable across participants whatever their amount of
                    # wake (older records without the column fall back to the every-epoch share)
                    pct_ref = (kept['pct_epochs_rejected_ref'].fillna(kept['pct_epochs_rejected'])
                               if 'pct_epochs_rejected_ref' in kept.columns else kept['pct_epochs_rejected'])
                    ax.bar(kept['file_id'].astype(str), pct_ref, color='#34495e')
                    ax.set_ylabel('% reference-stage epochs rejected'); ax.set_xlabel('participant')
                    ax.set_ylim(0, 100)
                    ax.set_title('% reference-stage epochs rejected per participant'
                                 + (f' ({n_out} excluded participant(s) not shown)' if n_out else ''))
                    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
                    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
                    display(widgets.Image(value=buf.getvalue(), format='png'))
                    L.save_report_html(reports_out / 'autoreject_database_report.html',
                                       'Automatic rejection - database summary',
                                       [('% reference-stage epochs rejected per participant', fig)],
                                       wrap_scroll(summary.to_html(index=False)))
                except Exception as ge:
                    print(f'⚠ global chart/report failed: {ge}')
        except Exception as e:
            print(f'Run error: {e}')


btn_run.on_click(run_autoreject)
display(widgets.VBox([participant_selector.widget, btn_run, widgets.HBox([prog, lbl_prog]), out_run]))
